# Machine Learning Model Deployment


- Add MLflow to the previous classification model

```mermaid
flowchart TD
    A[MLflow] --> B[Experiment]

    B --> C[Run 1]
    B --> D[Run 2]
    B --> E[Run 3]

    C --> C1[Metrics]
    C --> C2[Parameters]
    C --> C3[Artifacts]

    D --> D1[Metrics]
    D --> D2[Parameters]
    D --> D3[Artifacts]

    E --> E1[Metrics]
    E --> E2[Parameters]
    E --> E3[Artifacts]

    C3 --> F[Model Registry]
    D3 --> F
    E3 --> F

    F --> G[rf_model]

    G --> H[Version 1]
    G --> I[Version 2]
```


## Libs

In [ ]:
import os
from pathlib import Path
import sys
from tqdm import tqdm
import subprocess

ROOT_DIR = Path.cwd().parent
sys.path.append(str(ROOT_DIR))

import matplotlib.pyplot as plt
import seaborn as sns

from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.ml.feature import VectorAssembler
from pyspark.sql.types import StringType
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

import mlflow
import mlflow.spark
from mlflow.models import infer_signature
from graphics.graphs import plot_confusion_matrix

## Dataset

In [4]:
df = spark.read.csv('data/dataset_reduced_imdb.csv', header=True, inferSchema=True)

print(f'# Amostras: {df.count():_}')
df.limit(24).toPandas()

# Amostras: 1_650


,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres,averageRating,numVotes
0,tt22489120,movie,La Lucha: Getting Schooled in America,La Lucha: Getting Schooled in America,0,2023,\N,80,Documentary,9.9,62
1,tt21987706,movie,Dancing with Mom,Dancing with Mom,0,2022,\N,76,Documentary,9.7,69
2,tt9032398,movie,The Fire Cats,The Fire Cats: Save Something Small,0,2022,\N,80,Documentary,9.7,79
3,tt10703554,movie,The Book of Harth,The Book of Harth,0,2022,\N,62,Documentary,9.6,58
4,tt21062574,movie,Battle of Amritsar,Battle of Amritsar,0,2022,\N,156,Documentary,9.6,60
5,tt10218112,movie,Courageous Warriors Beauty from the Ashes,Courageous Warriors Beauty from the Ashes,0,2021,\N,91,Documentary,9.5,51
6,tt21110404,movie,Breaking la Vida,Breaking la Vida,0,2022,\N,93,Documentary,9.5,60
7,tt19879476,movie,War Dogs and I,War Dogs and I,0,2022,\N,60,Documentary,9.5,89
8,tt32250502,movie,The Years We Have Been Nowhere,The Years We Have Been Nowhere,0,2023,\N,80,Documentary,9.5,87
9,tt27133532,movie,Ambatukam: The Rise and Fall of Dreamybull,Ambatukam: The Rise and Fall of Dreamybull,0,2023,\N,45,Documentary,9.5,75


## Missing values

In [5]:
string_cols = [
    field.name
    for field in df.schema.fields
    if isinstance(field.dataType, StringType)
]

df.select([
    F.count(
        F.when(F.col(c) == r'\N', F.lit(1))
    ).alias(c)
    for c in string_cols
]).toPandas()

,tconst,titleType,primaryTitle,originalTitle,endYear,runtimeMinutes,genres
0,0,0,0,0,1650,155,0


In [6]:
df.limit(24).toPandas()

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres,averageRating,numVotes
0,tt22489120,movie,La Lucha: Getting Schooled in America,La Lucha: Getting Schooled in America,0,2023,\N,80,Documentary,9.9,62
1,tt21987706,movie,Dancing with Mom,Dancing with Mom,0,2022,\N,76,Documentary,9.7,69
2,tt9032398,movie,The Fire Cats,The Fire Cats: Save Something Small,0,2022,\N,80,Documentary,9.7,79
3,tt10703554,movie,The Book of Harth,The Book of Harth,0,2022,\N,62,Documentary,9.6,58
4,tt21062574,movie,Battle of Amritsar,Battle of Amritsar,0,2022,\N,156,Documentary,9.6,60
5,tt10218112,movie,Courageous Warriors Beauty from the Ashes,Courageous Warriors Beauty from the Ashes,0,2021,\N,91,Documentary,9.5,51
6,tt21110404,movie,Breaking la Vida,Breaking la Vida,0,2022,\N,93,Documentary,9.5,60
7,tt19879476,movie,War Dogs and I,War Dogs and I,0,2022,\N,60,Documentary,9.5,89
8,tt32250502,movie,The Years We Have Been Nowhere,The Years We Have Been Nowhere,0,2023,\N,80,Documentary,9.5,87
9,tt27133532,movie,Ambatukam: The Rise and Fall of Dreamybull,Ambatukam: The Rise and Fall of Dreamybull,0,2023,\N,45,Documentary,9.5,75


## Trata valores nulos de runtimeMinutes

In [ ]:
df = df.withColumn(
    'runtimeMinutes',
    F.expr('try_cast(runtimeMinutes as double)')
)

runtime_mean = df.select(
    F.mean('runtimeMinutes')
).first()[0]

if runtime_mean is not None:
    df = df.withColumn(
        'runtimeMinutes',
        F.coalesce(
            F.col('runtimeMinutes'),
            F.lit(runtime_mean)
        )
    )

df.select('runtimeMinutes').limit(10).toPandas()

,runtimeMinutes
0,80.0
1,76.0
2,80.0
3,62.0
4,156.0
5,91.0
6,93.0
7,60.0
8,80.0
9,45.0


## Define target

In [8]:
df = df.withColumn(
    'target',
    F.when(F.col('averageRating') > 7.5, 1).otherwise(0)
)

In [9]:
df.limit(5).toPandas()

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres,averageRating,numVotes,target
0,tt22489120,movie,La Lucha: Getting Schooled in America,La Lucha: Getting Schooled in America,0,2023,\N,80.0,Documentary,9.9,62,1
1,tt21987706,movie,Dancing with Mom,Dancing with Mom,0,2022,\N,76.0,Documentary,9.7,69,1
2,tt9032398,movie,The Fire Cats,The Fire Cats: Save Something Small,0,2022,\N,80.0,Documentary,9.7,79,1
3,tt10703554,movie,The Book of Harth,The Book of Harth,0,2022,\N,62.0,Documentary,9.6,58,1
4,tt21062574,movie,Battle of Amritsar,Battle of Amritsar,0,2022,\N,156.0,Documentary,9.6,60,1


In [10]:
df.orderBy(F.monotonically_increasing_id().desc()).limit(5).toPandas()

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres,averageRating,numVotes,target
0,tt23016770,movie,Chic & Classic: Meghan Markle,Chic & Classic: Meghan Markle,0,2022,\N,52.000000,Documentary,1.3,57,0
1,tt32636077,movie,Above and Below the Ground,Above and Below the Ground,0,2023,\N,86.000000,Documentary,1.4,78,0
2,tt14909626,movie,Don't Mess Up Our Race,Don't Mess Up Our Race,0,2021,\N,92.278261,Drama,1.4,71,0
3,tt27478726,movie,Red Flower,Gol-e Sorkh,0,2023,\N,76.000000,Comedy,2.1,61,0
4,tt16538780,movie,Aska Dair,Aska Dair,0,2022,\N,122.000000,Comedy,2.1,58,0


In [11]:
print('\nDistribuição do Target:')
df.groupBy('target').count().orderBy('target').toPandas()


Distribuição do Target:


,target,count
0,0,1276
1,1,374


## VectorAssembler das features

In [12]:
features_cols = ['numVotes', 'isAdult', 'startYear', 'runtimeMinutes']

assembler = VectorAssembler(
    inputCols=features_cols,
    outputCol='features'
)

ml_df = (
    assembler
    .transform(df)
)

ml_df.limit(10).toPandas()

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres,averageRating,numVotes,target,features
0,tt22489120,movie,La Lucha: Getting Schooled in America,La Lucha: Getting Schooled in America,0,2023,\N,80.0,Documentary,9.9,62,1,"[62.0, 0.0, 2023.0, 80.0]"
1,tt21987706,movie,Dancing with Mom,Dancing with Mom,0,2022,\N,76.0,Documentary,9.7,69,1,"[69.0, 0.0, 2022.0, 76.0]"
2,tt9032398,movie,The Fire Cats,The Fire Cats: Save Something Small,0,2022,\N,80.0,Documentary,9.7,79,1,"[79.0, 0.0, 2022.0, 80.0]"
3,tt10703554,movie,The Book of Harth,The Book of Harth,0,2022,\N,62.0,Documentary,9.6,58,1,"[58.0, 0.0, 2022.0, 62.0]"
4,tt21062574,movie,Battle of Amritsar,Battle of Amritsar,0,2022,\N,156.0,Documentary,9.6,60,1,"[60.0, 0.0, 2022.0, 156.0]"
5,tt10218112,movie,Courageous Warriors Beauty from the Ashes,Courageous Warriors Beauty from the Ashes,0,2021,\N,91.0,Documentary,9.5,51,1,"[51.0, 0.0, 2021.0, 91.0]"
6,tt21110404,movie,Breaking la Vida,Breaking la Vida,0,2022,\N,93.0,Documentary,9.5,60,1,"[60.0, 0.0, 2022.0, 93.0]"
7,tt19879476,movie,War Dogs and I,War Dogs and I,0,2022,\N,60.0,Documentary,9.5,89,1,"[89.0, 0.0, 2022.0, 60.0]"
8,tt32250502,movie,The Years We Have Been Nowhere,The Years We Have Been Nowhere,0,2023,\N,80.0,Documentary,9.5,87,1,"[87.0, 0.0, 2023.0, 80.0]"
9,tt27133532,movie,Ambatukam: The Rise and Fall of Dreamybull,Ambatukam: The Rise and Fall of Dreamybull,0,2023,\N,45.0,Documentary,9.5,75,1,"[75.0, 0.0, 2023.0, 45.0]"


## Split dataset into training and test

In [13]:
train_df, test_df = ml_df.randomSplit([0.8, 0.2], seed=42)

print(f'#Linhas treino: {train_df.count()}')
print(f'#Linhas de teste: {test_df.count()}')

#Linhas treino: 1363
#Linhas de teste: 287


### Export training and test datasets

In [14]:
train_df.toPandas().to_csv('data/train_df.csv', index=False)
test_df.toPandas().to_csv('data/test_df.csv', index=False)

In [15]:
test_df.groupBy('target').count().orderBy('target').toPandas()

,target,count
0,0,221
1,1,66


## MLFlow

### Start server

http://localhost:5000

In [ ]:
mlflow_process = subprocess.Popen(
    [
        'mlflow', 'server',
        '--backend-store-uri', 'sqlite:///mlflow.db',
        '--host', '127.0.0.1',
        '--port', '5000',
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

### Setup MLflow

In [19]:
!curl http://localhost:5000/health

OK

In [20]:
model_name = f'rf_model'

client = mlflow.MlflowClient()

def get_latest_version(model_name):
  model_version_infos = client.search_model_versions(f'name = "{model_name}"')
  return max([
    model_version_info.version for model_version_info in model_version_infos
  ])

In [ ]:
mlflow.set_tracking_uri('http://localhost:5000')
mlflow.set_registry_uri('http://localhost:5000')
mlflow.set_experiment('score_classifier_imdb')
mlflow.autolog(disable=False)

2026/08/30 18:05:51 WARNING mlflow.spark: With Pyspark >= 3.2, PYSPARK_PIN_THREAD environment variable must be set to false for Spark datasource autologging to work.
2026/08/30 18:05:51 WARNING mlflow.tracking.fluent: Exception raised while enabling autologging for pyspark: Exception while attempting to initialize JVM-side state for Spark datasource autologging. Note that Spark datasource autologging only works with Spark 3.0 and above. Please create a new Spark session with required Spark version and ensure you have the mlflow-spark JAR attached to your Spark session as described in https://mlflow.org/docs/latest/tracking/autolog.html#spark Exception:
'JavaPackage' object is not callable
2026/08/30 18:05:51 INFO mlflow.tracking.fluent: Autologging successfully enabled for pyspark.ml.


### Model registry

<img src="imgs/model_overview.png">

<img src="imgs/metrics.png">

## Para servidor MLflow

In [ ]:
mlflow_process.terminate()
mlflow_process.wait()

In [27]:
spark.stop()